In [1]:
import os
import joblib
import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", None)

import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    f1_score,
    recall_score,
    confusion_matrix,
    roc_curve,
    roc_auc_score,
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import plotly.graph_objects as go
import plotly.subplots as subplots

from gensim.models import KeyedVectors
import gensim.downloader as api

## Load Dataset

In [2]:
df = pd.read_csv("./data/IMDB Reviews.csv")
print(df.info())
print(df["sentiment"].value_counts(normalize=True))
print(df["review"].apply(lambda x: len(x)).value_counts(normalize=True))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB
None
sentiment
positive    0.5
negative    0.5
Name: proportion, dtype: float64
review
658     0.00192
665     0.00176
684     0.00166
667     0.00166
670     0.00166
         ...   
5048    0.00002
4355    0.00002
4236    0.00002
5452    0.00002
6620    0.00002
Name: proportion, Length: 4801, dtype: float64


## Data Preprocessing

In [3]:
%%time

# Data Cleaning


def clean_message(data):
    output = data.str.strip()
    output = output.str.lower()
    output = output.str.replace(r"<[^>]+>", "", regex=True)
    output = output.str.replace(r"http\S+|www\.\S+", "", regex=True)
    output = output.str.replace(r"[^a-zA-Z\s]+", "", regex=True)
    output = output.str.replace(r"\s+", " ", regex=True)
    return output


df["clean_text"] = clean_message(df["review"])

CPU times: user 6.71 s, sys: 121 ms, total: 6.83 s
Wall time: 7.73 s


In [4]:
%%time

# Tokenization and conversion to numerical representation

reviews = df["clean_text"].values

tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(reviews)
data = tokenizer.texts_to_sequences(reviews)
input_dim = 300
data = pad_sequences(data, maxlen=input_dim)

CPU times: user 11 s, sys: 65 ms, total: 11.1 s
Wall time: 11.2 s


In [5]:
# Train and Test Split

labels = df["sentiment"].map({"negative": 0, "positive": 1}).values

x_train, x_test_and_val, y_train, y_test_and_val = train_test_split(
    data, labels, test_size=0.1, random_state=42, shuffle=True, stratify=labels
)

x_val, x_test, y_val, y_test = train_test_split(
    x_test_and_val,
    y_test_and_val,
    test_size=0.5,
    random_state=42,
    shuffle=True,
    stratify=y_test_and_val,
)

## Load Pretrained Word2vec

In [6]:
%%time

# Load Embeddings

pretrained_model_path = "./artifacts/GoogleNews-vectors-negative300.bin"

try:
    word_vectors = KeyedVectors.load_word2vec_format(pretrained_model_path, binary=True)
except FileNotFoundError:
    word_vectors = api.load("word2vec-google-news-300")
    os.makedirs("./artifacts", exist_ok=True)
    word_vectors.save_word2vec_format(pretrained_model_path, binary=True)

CPU times: user 20.1 s, sys: 3.57 s, total: 23.7 s
Wall time: 24.8 s


In [7]:
# Define Embeddings Matrix

word_index = tokenizer.word_index
vocab_size = len(word_index) + 1
embedding_dim = 300

embedding_matrix = np.zeros((vocab_size, embedding_dim))

In [8]:
# Initialise Embedding Matrix

for word, index in word_index.items():
    if word not in word_vectors:
        embedding_matrix[index] = np.random.uniform(
            low=-0.25, high=0.25, size=(embedding_dim,)
        )
    else:
        embedding_matrix[index] = word_vectors[word]

## Build and Train LSTM Model

In [9]:
# Initialise LSTM Model

model = Sequential(
    [
        Embedding(
            input_dim=vocab_size,
            output_dim=embedding_dim,
            weights=[embedding_matrix],
            trainable=True,
        ),
        Dropout(0.5),
        Bidirectional(LSTM(48)),
        Dropout(0.6),
        Dense(1, activation="sigmoid"),
    ]
)

model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

In [10]:
%%time

# Train Model

history = model.fit(
    x_train,
    y_train,
    epochs=10,
    batch_size=256,
    validation_data=(x_val, y_val),
    callbacks=EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
)

Epoch 1/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 24s 100ms/step - accuracy: 0.6737 - loss: 0.5821 - val_accuracy: 0.8520 - val_loss: 0.3668
Epoch 2/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 17s 97ms/step - accuracy: 0.8923 - loss: 0.2866 - val_accuracy: 0.8872 - val_loss: 0.2756
Epoch 3/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 17s 99ms/step - accuracy: 0.9281 - loss: 0.2026 - val_accuracy: 0.8932 - val_loss: 0.2854
Epoch 4/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 17s 96ms/step - accuracy: 0.9545 - loss: 0.1353 - val_accuracy: 0.8924 - val_loss: 0.2883
CPU times: user 28 s, sys: 3.93 s, total: 31.9 s
Wall time: 1min 15s


In [11]:
# Save model and predict values

joblib.dump(model, "./artifacts/sentiment_classifier.pkl")

y_pred = model.predict(x_test)
y_pred_binary = [1 if x >= 0.5 else 0 for x in y_pred]

79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step


In [12]:
## Metrics

print(f"Accuracy: {round(100 * accuracy_score(y_test, y_pred_binary), 2)}")
print(f"Precision: {round(100 * precision_score(y_test, y_pred_binary), 2)}")
print(f"Recall: {round(100 * recall_score(y_test, y_pred_binary), 2)}")
print(f"F1: {round(100 * f1_score(y_test, y_pred_binary), 2)}")
df = pd.DataFrame(
    confusion_matrix(y_test, y_pred_binary),
    index=["Actual Negative", "Actual Positive"],
    columns=["Predicted Negative", "Predicted Positive"],
)
print(f"Confusion matrix:\n{df}")

Accuracy: 89.52
Precision: 91.17
Recall: 87.52
F1: 89.31
Confusion matrix:
                 Predicted Negative  Predicted Positive
Actual Negative                1144                 106
Actual Positive                 156                1094


In [13]:
## ROC curve

auc_score = roc_auc_score(y_test, y_pred)
fpr, tpr, thresholds = roc_curve(y_test, y_pred)
print(f"AUC: {auc_score:.4f}")

trace1 = go.Scatter(x=fpr, y=tpr, name=f"ROC")
trace2 = go.Scatter(x=[0, 1], y=[0, 1], name="Reference Line", line={"dash": "dash"})

layout = {
    "xaxis_title": "False Positive Rate",
    "yaxis_title": "True Positive Rate",
    "title": "ROC Curve",
    "width": 800,
    "height": 680,
}
fig = go.Figure(data=[trace1, trace2], layout=layout)

fig.show()

AUC: 0.9593
